# Lichess — balanced 1+0 / 5+0 / 10+0 collection

**Goal:** Collect **30,000 games per time control** from the October 2026 official rated-game archive.

This notebook executes the included Python collector. It streams the compressed archive without saving the full 31 GB, and saves original PGN games, game metadata and a manifest locally.

**Important:** Streaming mode reads games from the start of the month until each quota is reached; it is not a random sample of the entire month. For whole-month sampling, use the script's `local_reservoir` mode after downloading the archive.


## 1. Install dependencies

From the project folder, run `python3 -m venv .venv` and `.venv/bin/python -m pip install -r requirements.txt` once. Then install the notebook kernel with `.venv/bin/python -m ipykernel install --user --name lichess-sampling --display-name "Lichess sampling (Python 3)"`, select that kernel in VS Code, and run this cell to verify it. The internet connection is needed on your own machine.


In [ ]:
import sys
from pathlib import Path

if not Path(sys.executable).as_posix().endswith("/.venv/bin/python"):
    raise RuntimeError(
        "Select the 'Lichess sampling (Python 3)' kernel from the project .venv first."
    )

print(f"Using {sys.executable}")


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## 2. Settings

Open **`02_lichess_stratified_sample.py`** in the same folder and edit the settings at the top:

- `QUOTA_PER_CONTROL = 100` for a quick pilot; set **`30_000`** for the full sample.
- `MODE = "stream"` reads the remote archive and stops once quotas are met.
- `MODE = "local_reservoir"` scans a *fully downloaded* month and produces a more representative whole-month random sample.

The collection only uses PGN headers and clock annotations; **it does not calculate Stockfish labels yet**.


In [ ]:
# Execute the collector in this notebook's working directory.
# For the full sample, first verify QUOTA_PER_CONTROL = 30000 in the .py file.
%run 02_lichess_stratified_sample.py

## 3. Outputs

Generated under `data/lichess_2026-10_balanced/`:

- `games_1m.pgn`, `games_5m.pgn`, `games_10m.pgn`: original PGN records
- `games_metadata.csv`: game IDs, dates, Elo, time controls and clock counts
- `dataset_manifest.json`: exact source, filters, and achieved counts

**Next stage:** use `python-chess` to validate each game, reconstruct *pre-move* clocks, and calculate position features. Evaluate a controlled subset of positions with local Stockfish before attempting millions of engine searches.
